# 🔦 TrustyAI
모델을 학습한 후 프로덕션에 배포하기 전에 테스트함으로써 현재 사용 가능한 데이터에 대한 모델의 성능을 잘 이해할 수 있습니다.  
하지만 현실 세계는 매우 복잡하고 항상 변하고 있으므로, 어제 당연하다고 생각했던 것이 오늘은 완전히 다를 수 있습니다. 따라서 모델이 프로덕션에 배포되어 실시간 데이터를 보게 될 때 세계의 변화가 모델의 정확한 예측 능력에 영향을 미치는지 모니터링해야 합니다.  
모델이나 데이터가 프로덕션에 배포된 후 이상하게 동작하기 시작하면 나타날 수 있는 주요 지표들을 추적하려고 합니다.  
이를 위해 TrustyAI라는 도구가 있어서 공정성과 드리프트 탐지를 위한 메트릭을 생성하는데 도움이 될 수 있습니다. TrustyAI를 사용하면 실시간 데이터가 학습 데이터셋을 생성할 때 예상했던 것과 다른지 판단할 수 있습니다. 또한 우리의 모델이 초기에 예상했던 것과 실제 데이터에 다르게 대응하는지를 측정할 수 있습니다.

### onnxruntime 설치
TrustyAI에 전송할 데이터를 생성할 때 나중에 필요합니다.

`pip`에서 오류가 발생해도 걱정하지 마세요. 어쨌든 잘 작동할 것입니다.

In [ ]:
!pip -q install onnxruntime model-registry==0.2.15

### 사용자 토큰 받기
TrustyAI 서비스에 요청을 보낼 수 있도록 사용자 토큰을 제공해야 합니다.
사용자 토큰을 받으려면:
1. OpenShift 콘솔로 이동합니다
2. 사용자 이름이 표시된 오른쪽 상단의 드롭다운을 클릭합니다
3. "로그인 명령 복사" 선택합니다
4. 로그인합니다
5. 첫 번째 코드 상자에서 `token` 다음의 부분을 선택합니다. `sha256~.....` 같은 형태여야 합니다

### 모델 버전 받기

이전에 실행한 모델 학습 파이프라인에서 몇 가지 아티팩트를 가져올 것이고, 이를 실행한 파이프라인을 추적하는 모델 레지스트리를 활용하여 수행할 것입니다.  
이를 위해 어떤 모델 버전에 관심이 있는지 지정해야 합니다.  
*userX*-prod-registry라는 모델 레지스트리로 이동하여 git 해시처럼 보이는 **첫 번째** 모델 버전을 가져옵니다.

### 추론 엔드포인트 받기

추론 엔드포인트를 받으려면:
1. RHOAI 왼쪽 메뉴의 모델 서빙으로 이동합니다
2. *userX*-test 프로젝트를 선택합니다
3. **외부 경로** 를 추론 엔드포인트로 복사합니다

In [ ]:
token = "ENTER-USER-TOKEN"
model_version = "ENTER-YOUR-MODEL-VERSION"
cluster_domain = "ENTER-YOUR-CLUSTER-DOMAIN"
infer_endpoint = "ENTER-YOUR-INFERENCE-ENDPOINT"

model_name = "jukebox"

In [ ]:
import pandas as pd
import pickle
import json
import onnxruntime as rt
import numpy as np
import onnx

import requests
from urllib.parse import urljoin
from fetch_artifacts_from_registry import fetch_artifacts_from_registry

### 데이터
학습 데이터를 TrustyAI 서비스로 전송하여 학습 데이터를 우리의 추론 요청에서 들어오는 새로운 데이터와 비교할 수 있게 합니다.  
일반적인 용도를 위해서는 5000개 샘플로 제한하고 있지만, 실제 사용 사례에서는 전체 학습 데이터를 전송하거나 그 분포를 적절하게 나타내는 일부를 전송할 것입니다.

In [ ]:
namespace_file_path =\
    '/var/run/secrets/kubernetes.io/serviceaccount/namespace'
with open(namespace_file_path, 'r') as namespace_file:
    current_namespace = namespace_file.read()
username = current_namespace.split("-")[0]

In [ ]:
artifacts = ["preprocess-data/train_data.pkl", "convert-keras-to-onnx/onnx_model.onnx", "preprocess-data/test_data.pkl"]
pipeline_namespace = f"{username}-toolings"
model_registry_url = f"https://{username}-prod-registry-rest.{cluster_domain}"

In [ ]:
saved_files = fetch_artifacts_from_registry(
    token,
    artifacts,
    pipeline_namespace,
    model_registry_url,
    model_name,
    model_version,
    username,
)

In [ ]:
X_train = pd.read_pickle(saved_files['preprocess-data/train_data.pkl'])[0][:5000]

우리는 데이터의 예측값도 얻어서 나중에 드리프트하기 시작하는지 확인할 수 있습니다.

In [ ]:
sess = rt.InferenceSession(saved_files["convert-keras-to-onnx/onnx_model.onnx"], providers=rt.get_available_providers())
data_dict = {name: X_train[[name]].to_numpy().astype(np.float32) for name in X_train.columns}
output_name = sess.get_outputs()[0].name
y_pred_temp = sess.run([output_name], data_dict)

모든 데이터를 준비한 후 TrustyAI가 기대하는 특정 방식으로 구조화합니다.  
특히, 데이터에 data_tag를 추가하여 다양한 반복이나 다양한 목적으로 사용되는 데이터를 추적할 수 있습니다.

In [ ]:
training_data = {
    "model_name": model_name,
    "data_tag": "TRAINING",
    "request": {
        "inputs": [ 
           {
                "name": name,
                "shape": np.shape(data_dict[name]),
                "datatype": "FP32",
                "data": data_dict[name].tolist()
            }
            for name in data_dict.keys()
        ]
    },
    "response": {
        "model_name": model_name,
        "model_version": "1",
        "outputs": [
            {
                "name": output_name,
                "datatype": "FP32",
                "shape": np.shape(y_pred_temp[0]),
                "data": y_pred_temp[0].tolist()
            }
        ]
    }
}


### REST 요청으로 TrustyAI와 상호작용
REST 요청을 통해 우리의 TrustyAI 서비스와 상호작용할 것입니다.  
그 전에 우리가 요청을 보낼 위치를 알 수 있도록 TrustyAI 서비스 경로를 `base_url`로 추가해야 합니다.

In [ ]:
base_url = f"http://trustyai-service.{username}-test.svc.cluster.local"
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

먼저 준비한 데이터를 업로드합니다.

In [ ]:
# 데이터 업로드
endpoint = "data/upload"
url = urljoin(base_url, endpoint)
response = requests.post(url, headers=headers, json=training_data)
print(response.text)

이제 우리는 드리프트 탐지 메트릭을 구독할 수 있으며, 이는 우리의 TrustyAI 서비스가 지속적으로 드리프트(특히 meanshift) 메트릭을 발행하게 합니다.  

meanshift 메트릭은 학습 데이터의 분포가 우리가 보낸 새로운 데이터와 얼마나 다르게 보이는지를 추적합니다.  

각 개별 입력 및 출력 특성에 대해 0과 1 사이의 "p-value"를 얻게 됩니다. p-value가 1.0인 것은 학습 데이터와 테스트 데이터가 동일한 분포에서 나왔을 가능성이 매우 높음을 나타내는 반면, p-value < 0.05는 학습 세트와 테스트 세트 간의 통계적으로 유의미한 드리프트를 나타냅니다.

In [ ]:
# Meanshift 모니터링
endpoint = "/metrics/drift/meanshift/request"
url = urljoin(base_url, endpoint)

payload = {
    "modelId": model_name,
    "referenceTag": "TRAINING"
}

response = requests.post(url, headers=headers, json=payload)
print(response.text)

모든 것이 올바르게 보이는지 확인하기 위해 TrustyAI 서비스에 현재 설정이 어떻게 보이는지 물을 수 있습니다.  
우리는 구독한 메트릭과 추가된 데이터에 대한 정보를 다시 받을 것입니다.  

In [ ]:
# 등록한 내용 확인
endpoint = "/info"
url = urljoin(base_url, endpoint)
response = requests.get(url, headers=headers)
print(response.text)

### 퀴즈 시간 🤓

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../.dontlookhere/'))
from quiz4 import *

In [ ]:
quiz_monitoring()

In [ ]:
quiz_drift()

### 요청 보내기
마지막으로 모델 서버에 단일 요청을 보내야 TrustyAI이 메트릭을 발행하기 시작합니다. 이는 학습 데이터와 비교할 최소한 하나의 추론 데이터 포인트를 가져야 합니다.

In [ ]:
infer_url = f"{infer_endpoint}/v2/models/{model_name}/infer"

def rest_request(data):
    json_data = {
        "inputs": [
           {
                "name": name,
                "shape": [1, 1],
                "datatype": "FP32",
                "data": [data[name][0].tolist()]
            }
            for name in data.keys()
        ]
    }

    print(json_data)
    response = requests.post(infer_url, json=json_data, verify=True)
    response_dict = response.json()
    print(response_dict)
    return response_dict['outputs'][0]['data']


prediction = rest_request(data_dict)

또한 어떤 특성의 평균값을 모니터링하여 시간에 따라 어떻게 변하는지 볼 수 있습니다.

In [ ]:
# 평균값 모니터링
endpoint = "/metrics/identity/request"
url = urljoin(base_url, endpoint)

payload = {
    "modelId": model_name,
    "columnName": "duration_ms",
    "batchSize": 256,
}

response = requests.post(url, headers=headers, json=payload)
print(response.text)

다음 노트북으로 가서 드리프트를 생성하고 OpenShift UI에서 메트릭을 관찰합시다 👉 [jukebox/4-metrics/2-introducing_drift.ipynb](2-introducing_drift.ipynb)